# Re-run: agreement (RQ1)

This notebook recomputes the headline numbers of RQ1 from the benchmark repository's committed outputs:
`out/leaderboard.json` (every model's QWK on the board's 100 answers per task, with its share of the
raters' agreement), `out/baselines.json` (the trained scorers) and `essays-finetune/out/essays-finetune.json`
(the fine-tuning study). It counts from the intervals themselves rather than reading verdict strings. It
needs only Python; the gradings behind these files are not published (the harness is held back), so the
QWKs are recomputed by the scripts that wrote these files, not here.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "ai-grading-benchmark-2026"   # set when the report is built
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


SOURCE_ENV, SOURCE_NAME = "GBENCH_DIR", "ai-grading-benchmark"   # set when a release's code is another repository


def source_repo():
    """The repository a release's code and committed outputs are published in (a release whose paper cites
    it rather than a copied code package): $SOURCE_ENV, or a checkout named SOURCE_NAME beside this
    notebook or beside a folder above it."""
    here = Path.cwd()
    places = [Path(os.environ[SOURCE_ENV])] if SOURCE_ENV and os.environ.get(SOURCE_ENV) else []
    places += [p / SOURCE_NAME for p in [here, *here.parents]]
    for d in places:
        if d.is_dir() and (d / "out").is_dir():
            return d
    raise FileNotFoundError(
        f"Could not find the {SOURCE_NAME} repository this release's code is published in. Clone it at the "
        f"commit the release file names, then set the environment variable {SOURCE_ENV} to the clone, or "
        f"run this notebook from a folder beside it.")


def load_source_json(relpath):
    """A committed output of that repository, by its path there (cited in the paper as code/<path>)."""
    return json.loads((source_repo() / relpath).read_text(encoding="utf-8"))


print("Source repository:", shown(source_repo()))

Source repository: ../../../../../../ai-grading-benchmark


In [2]:
from decimal import Decimal, ROUND_HALF_UP

def p(x, d=3, sign=False):
    """Round half-up once from the unrounded value, as the paper prints it."""
    v = Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP)
    return f"{v:+.{d}f}" if sign else f"{v:.{d}f}"

lb = load_source_json("out/leaderboard.json")["rows"]
by = {}
for r in lb:
    by.setdefault(r["task"], []).append(r)
assert all(len(v) == 22 for v in by.values()), {k: len(v) for k, v in by.items()}

for task, lo, hi in (("mohler", "0.379", "0.680"), ("code", "0.268", "0.879"), ("asap2", "0.155", "0.727")):
    q = [r["qwk"]["estimate"] for r in by[task]]
    print(task, "QWK from", p(min(q)), "to", p(max(q)))
    assert (p(min(q)), p(max(q))) == (lo, hi)

mohler QWK from 0.379 to 0.680
code QWK from 0.268 to 0.879
asap2 QWK from 0.155 to 0.727


**Against the raters.** On code, a model is shown to agree with the raters more than they agree with each
other when its share of the ceiling has an interval wholly above 100%, and less when wholly below. On
essays, the ceiling is the published weighted κ 0.745.

In [3]:
code = by["code"]
above = sum(r["share_of_ceiling"]["ci95"][0] > 1 for r in code)
below = sum(r["share_of_ceiling"]["ci95"][1] < 1 for r in code)
print("code: above the raters", above, "below", below)
assert (above, below) == (4, 14)
ess = by["asap2"]
ceiling = ess[0]["published_ceiling"]["value"]
whole_below = sum(r["qwk"]["ci95"][1] < ceiling for r in ess)
print("essays: published agreement", ceiling, "; models whose whole interval lies below it:", whole_below)
assert ceiling == 0.745 and whole_below == 18
assert not any(r["qwk"]["estimate"] >= ceiling for r in ess)
mohler_below = [r["model"] for r in by["mohler"] if r["share_of_ceiling"]["ci95"][1] < 1]
print("short answers: shown to agree less than the raters:", mohler_below)
assert mohler_below == ["meta-llama/llama-3.2-3b-instruct"]

code: above the raters 4 below 14
essays: published agreement 0.745 ; models whose whole interval lies below it: 18
short answers: shown to agree less than the raters: ['meta-llama/llama-3.2-3b-instruct']


**Trained scorers.** A scorer that uses only the essay's length, and DeBERTa fine-tuned on the corpus's
graded essays, set against the 22 prompted models on the same 100 essays; then the paired comparison of
DeBERTa with the best prompted arm on 500 held-out essays.

In [4]:
bl = {(r["model"], r["task"]): r for r in load_source_json("out/baselines.json")["rows"]}
length = bl[("baseline: length", "asap2")]["qwk"]["estimate"]
deberta = bl[("baseline: deberta-ft", "asap2")]["qwk"]["estimate"]
beaten = sum(r["qwk"]["estimate"] < length for r in ess)
print("length-only essay QWK", p(length), "is higher than", beaten, "of 22 models; DeBERTa", p(deberta))
assert p(length) == "0.636" and beaten == 16 and p(deberta) == "0.809"
assert deberta > max(r["qwk"]["estimate"] for r in ess)
c2 = load_source_json("essays-finetune/out/essays-finetune.json")["comparisons"]["C2"]
print("DeBERTa minus Claude Opus 5.5 with six examples:", p(-c2["estimate"]), [p(-c2["ci95"][1]), p(-c2["ci95"][0])])
assert (p(-c2["estimate"]), p(-c2["ci95"][1]), p(-c2["ci95"][0])) == ("0.095", "0.059", "0.130")
assert c2["ci95"][1] < 0   # the direction: the prompted arm minus DeBERTa lies below 0

length-only essay QWK 0.636 is higher than 16 of 22 models; DeBERTa 0.809
DeBERTa minus Claude Opus 5.5 with six examples: 0.095 ['0.059', '0.130']
